# Level 1: benchmarks

Pushes 10 benchmark questions whose SQL is the ground truth. Run the same set in the Genie UI. Scoring is exact. TIMEOUT, NO_ANSWER, and ERROR count as FAIL.

**Beta:** the eval-runs API is Beta. If the call is disabled, run the suite in the Benchmarks tab instead.


## Bind config and agent ids


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)

ids = spark.table(cfg.table("agent_ids")).collect()[0]
curated_id = ids["curated_space_id"]
print("curated_space_id", curated_id)


## Recompute ground-truth SQL live

Each benchmark SQL must run against your tables before you trust a score.


In [ ]:
import json
from pathlib import Path

raw = (root / "level-1-foundations" / "templates" / "agent_curated.json").read_text()
raw = raw.replace("${catalog}", cfg.catalog).replace("${schema}", cfg.schema)
curated = json.loads(raw)
rows = []
for item in curated["benchmarks"]:
    sql = item["sql"][0]
    result = spark.sql(sql)
    n = result.count()
    preview = [r.asDict(recursive=True) for r in result.limit(5).collect()]
    rows.append((item["id"], item["question"][0], n, preview))
    print(item["id"], "rows", n, item["question"][0])
assert len(rows) == 10
print("PASS 10 ground-truth queries")


## Optional eval-runs API

Starts a Beta eval-run on the curated agent. If the workspace returns FEATURE_DISABLED, use the UI Benchmarks tab.


In [ ]:
from shared.lib.genie_client import GenieClientError

client = GenieClient(warehouse_id=cfg.warehouse_id)
try:
    run = client.create_eval_run(curated_id)
    run_id = run.get("eval_run_id") or run.get("id")
    print("eval-run started", run_id)
    finished = client.wait_eval_run(curated_id, run_id)
    print("eval-run status", finished.get("status") or finished.get("state"))
    results = client.list_eval_results(curated_id, run_id)
    spark.createDataFrame(
        [(json.dumps(results)[:200000],)],
        ["payload"],
    ).write.mode("overwrite").saveAsTable(cfg.table("benchmark_runs"))
    print("wrote", cfg.table("benchmark_runs"))
except GenieClientError as exc:
    print("SKIP eval-runs API:", str(exc)[:500])
    print("Fallback: open the agent Benchmarks tab, run all 10, and aim for 10/10.")
